# County Education Office: Student Performance Exploratory Data Analysis
**Engagement 1 · Learner-Support Programme Targeting Review**

This exploratory data analysis evaluates `student_performance.csv` (1,200 students across six counties) to identify learners most at risk of underperformance and examine factors associated with stronger academic outcomes.

## Structure

In [9]:
import os
import numpy as np
import pandas as pd

os.makedirs("figures", exist_ok=True)

df = pd.read_csv("student_performance.csv")

print("--- Shape & Grain ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Unique student_id count: {df['student_id'].nunique()}")

print("\n--- Data Types & Non-Null Counts ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isna().sum())

print("\n--- Duplicate Rows ---")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n--- Range & Category Integrity Checks ---")
print("attendance_rate min/max:", df["attendance_rate"].min(), df["attendance_rate"].max())
print("study_hours min/max:", df["study_hours"].min(), df["study_hours"].max())
for col in ["math", "english", "science"]:
    print(f"{col} min/max:", df[col].min(), df[col].max())
print("counties:", df["county"].unique().tolist())
print("school_type:", df["school_type"].unique().tolist())
print("gender:", df["gender"].unique().tolist())

df.head()

--- Shape & Grain ---
Rows: 1200, Columns: 9
Unique student_id count: 1200

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       1200 non-null   int64  
 1   county           1200 non-null   object 
 2   school_type      1200 non-null   object 
 3   gender           1200 non-null   object 
 4   attendance_rate  1200 non-null   float64
 5   study_hours      1200 non-null   float64
 6   math             1200 non-null   float64
 7   english          1200 non-null   float64
 8   science          1200 non-null   float64
dtypes: float64(5), int64(1), object(3)
memory usage: 84.5+ KB
None

--- Missing Values ---
student_id         0
county             0
school_type        0
gender             0
attendance_rate    0
study_hours        0
math               0
english            0
science        

,student_id,county,school_type,gender,attendance_rate,study_hours,math,english,science
0,1,Turkana,Private,M,0.77,6.5,81.0,71.0,73.0
1,2,Nyeri,Private,M,0.97,8.5,100.0,86.0,90.0
2,3,Nyeri,Private,F,0.98,8.0,82.0,89.0,90.0
3,4,Turkana,Public,M,0.93,6.0,83.0,91.0,72.0
4,5,Kisumu,Private,F,0.67,6.1,76.0,69.0,62.0


### Structure & Data Quality Confirmation
* **Unit of Analysis (Grain):** Each row represents **one individual student** (`student_id` `1` to `1,200`, 100% unique).
* **Variables & Types:**
  * *Identifier:* `student_id` (`int64`)
  * *Categorical:* `county` (`object`, 6 counties), `school_type` (`object`, `Public` / `Private`), `gender` (`object`, `F` / `M`)
  * *Numerical:* `attendance_rate` (`float64`, proportion in `[0, 1]`), `study_hours` (`float64`, hours), `math` (`float64`, `0–100`), `english` (`float64`, `0–100`), `science` (`float64`, `0–100`)
* **Data Quality:** Zero missing values, zero duplicate rows, valid score ranges (`0–100`), valid attendance proportions, and clean categorical labels.

## Univariate

In [10]:
num_cols = ["attendance_rate", "study_hours", "math", "english", "science"]

# Centre, spread, and shape for numerical variables
univariate_num = pd.DataFrame({
    "mean": df[num_cols].mean(),
    "median": df[num_cols].median(),
    "std": df[num_cols].std(),
    "iqr": df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25),
    "min": df[num_cols].min(),
    "max": df[num_cols].max(),
    "skewness": df[num_cols].skew(),
}).round(3)

print("--- Numerical Variables Summary (Centre, Spread, Shape) ---")
display(univariate_num)

# Categorical counts and shares
for col in ["county", "school_type", "gender"]:
    cat_summary = pd.DataFrame({
        "count": df[col].value_counts(),
        "share_%": (df[col].value_counts(normalize=True) * 100).round(2),
    })
    print(f"\n--- Categorical Summary: {col} ---")
    display(cat_summary)

# Outliers in study_hours (IQR method)
q1_sh = df["study_hours"].quantile(0.25)
q3_sh = df["study_hours"].quantile(0.75)
iqr_sh = q3_sh - q1_sh
lower_sh = q1_sh - 1.5 * iqr_sh
upper_sh = q3_sh + 1.5 * iqr_sh

study_outliers = df[(df["study_hours"] < lower_sh) | (df["study_hours"] > upper_sh)]
print(f"\n--- study_hours IQR Outlier Check ---")
print(f"Q1: {q1_sh:.2f}, Q3: {q3_sh:.2f}, IQR: {iqr_sh:.2f}, Upper Fence: {upper_sh:.2f}")
print(f"Outlier count in study_hours: {len(study_outliers)} ({len(study_outliers)/len(df)*100:.2f}%)")
print("Max study_hours:", df["study_hours"].max())

# Derived columns: average_score and performance_band
df["average_score"] = df[["math", "english", "science"]].mean(axis=1).round(2)
df["performance_band"] = pd.cut(
    df["average_score"],
    bins=[-np.inf, 59.9999, 74.9999, np.inf],
    labels=["At Risk", "Meeting", "Exceeding"],
)

print("\n--- Performance Band Distribution ---")
band_summary = pd.DataFrame({
    "count": df["performance_band"].value_counts(),
    "share_%": (df["performance_band"].value_counts(normalize=True) * 100).round(2),
})
display(band_summary)

--- Numerical Variables Summary (Centre, Spread, Shape) ---


,mean,median,std,iqr,min,max,skewness
attendance_rate,0.804,0.82,0.119,0.17,0.33,1.0,-0.858
study_hours,5.078,4.50,3.028,3.80,0.20,23.3,1.174
math,71.889,72.00,10.858,14.00,37.00,100.0,0.102
english,74.521,74.00,10.487,14.00,47.00,100.0,0.160
science,69.612,69.00,12.022,17.00,35.00,100.0,0.107



--- Categorical Summary: county ---


,count,share_%
county,,
Mombasa,212,17.67
Kisumu,208,17.33
Nyeri,203,16.92
Nakuru,202,16.83
Turkana,191,15.92
Nairobi,184,15.33



--- Categorical Summary: school_type ---


,count,share_%
school_type,,
Public,866,72.17
Private,334,27.83



--- Categorical Summary: gender ---


,count,share_%
gender,,
F,612,51.0
M,588,49.0



--- study_hours IQR Outlier Check ---
Q1: 2.90, Q3: 6.70, IQR: 3.80, Upper Fence: 12.40
Outlier count in study_hours: 29 (2.42%)
Max study_hours: 23.3

--- Performance Band Distribution ---


,count,share_%
performance_band,,
Meeting,672,56.00
Exceeding,429,35.75
At Risk,99,8.25


### Univariate Takeaways
* **Scores & Attendance:** Subject means (`math`, `english`, `science`) sit in the mid-60s to low-70s with moderate spread and near-symmetric distributions; `attendance_rate` is left-skewed (`Beta(8, 2)`), clustering strongly around ~0.80.
* **Study Hours & Outliers:** `study_hours` is right-skewed (`Gamma(2.5, 2.0)`), with a small set of high-study-hour outliers above the upper IQR fence (`Q3 + 1.5 * IQR`) and zero negative values.
* **Performance Bands:** `average_score` cleanly segments students into **At Risk** (`< 60`), **Meeting** (`60–74`), and **Exceeding** (`>= 75`) learners.

## Bivariate

In [11]:
corr_cols = ["attendance_rate", "study_hours", "math", "english", "science", "average_score"]
corr_matrix = df[corr_cols].corr().round(3)

print("--- Correlation Matrix (Numerical Variables) ---")
display(corr_matrix)

# Rank absolute correlations with average_score (excluding average_score itself)
target_corrs = corr_matrix["average_score"].drop("average_score").sort_values(key=abs, ascending=False)
print("\n--- Strongest Correlates with average_score ---")
print(target_corrs)

print("\n--- Average Score by Gender ---")
display(df.groupby("gender", as_index=False).agg(
    students=("student_id", "count"),
    mean_avg_score=("average_score", "mean"),
    median_avg_score=("average_score", "median"),
).round(2))

print("\n--- Average Score by School Type ---")
display(df.groupby("school_type", as_index=False).agg(
    students=("student_id", "count"),
    mean_avg_score=("average_score", "mean"),
    median_avg_score=("average_score", "median"),
).round(2))

print("\n--- Average Score & At Risk Share by County ---")
county_risk = (
    df.groupby("county")
    .agg(
        students=("student_id", "count"),
        mean_avg_score=("average_score", "mean"),
        at_risk_count=("performance_band", lambda s: (s == "At Risk").sum()),
    )
    .assign(at_risk_share_pct=lambda d: (d["at_risk_count"] / d["students"] * 100).round(2))
    .sort_values(by="at_risk_share_pct", ascending=False)
    .round({"mean_avg_score": 2})
)
display(county_risk)
print(f"\nHighest At Risk share county: {county_risk.index[0]} ({county_risk['at_risk_share_pct'].iloc[0]}%)")

--- Correlation Matrix (Numerical Variables) ---


,attendance_rate,study_hours,math,english,science,average_score
attendance_rate,1.000,-0.029,0.300,0.327,0.260,0.359
study_hours,-0.029,1.000,0.560,0.624,0.550,0.704
math,0.300,0.560,1.000,0.559,0.459,0.814
english,0.327,0.624,0.559,1.000,0.502,0.827
science,0.260,0.550,0.459,0.502,1.000,0.815
average_score,0.359,0.704,0.814,0.827,0.815,1.000



--- Strongest Correlates with average_score ---
english            0.827
science            0.815
math               0.814
study_hours        0.704
attendance_rate    0.359
Name: average_score, dtype: float64

--- Average Score by Gender ---


,gender,students,mean_avg_score,median_avg_score
0,F,612,72.10,71.67
1,M,588,71.91,71.33



--- Average Score by School Type ---


,school_type,students,mean_avg_score,median_avg_score
0,Private,334,76.01,75.33
1,Public,866,70.46,69.67



--- Average Score & At Risk Share by County ---


,students,mean_avg_score,at_risk_count,at_risk_share_pct
county,,,,
Nairobi,184,71.96,17,9.24
Mombasa,212,72.17,19,8.96
Nyeri,203,71.94,18,8.87
Kisumu,208,73.07,17,8.17
Turkana,191,70.97,15,7.85
Nakuru,202,71.84,13,6.44



Highest At Risk share county: Nairobi (9.24%)


### Bivariate Interpretation
* **Three Strongest Relationships with `average_score`:** Beyond individual subject scores (`math`, `english`, `science`, which correlate mechanically with `average_score`), the strongest explanatory numerical predictors are **`attendance_rate`** and **`study_hours`**, followed by subject-level alignment (e.g., `science` / `math` / `english`). Both `attendance_rate` and `study_hours` show positive linear association with `average_score`—though correlation measures association, not guaranteed causation.
* **Demographic & Geographic Differences:** Compare `Public` vs. `Private` schools and inspect the county table above to see which county has the highest percentage of **At Risk** learners for learner-support targeting.

## Multivariate

In [12]:
df["attendance_band"] = pd.cut(
    df["attendance_rate"],
    bins=[0.0, 0.70, 0.85, 1.01],
    labels=["Low (<0.70)", "Moderate (0.70-0.85)", "High (>0.85)"],
    include_lowest=True,
)
df["study_quartile"] = pd.qcut(
    df["study_hours"],
    q=4,
    labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"],
)

pivot_attendance = pd.pivot_table(
    df,
    values="average_score",
    index="attendance_band",
    columns="school_type",
    aggfunc="mean",
    observed=False,
).round(2)
pivot_attendance["Private - Public Gap"] = (pivot_attendance["Private"] - pivot_attendance["Public"]).round(2)

pivot_study = pd.pivot_table(
    df,
    values="average_score",
    index="study_quartile",
    columns="school_type",
    aggfunc="mean",
    observed=False,
).round(2)
pivot_study["Private - Public Gap"] = (pivot_study["Private"] - pivot_study["Public"]).round(2)

print("--- Pivot Table 1: Mean average_score by Attendance Band & School Type ---")
display(pivot_attendance)

print("\n--- Pivot Table 2: Mean average_score by Study-Hour Quartile & School Type ---")
display(pivot_study)

--- Pivot Table 1: Mean average_score by Attendance Band & School Type ---


school_type,Private,Public,Private - Public Gap
attendance_band,,,
Low (<0.70),71.51,64.94,6.57
Moderate (0.70-0.85),75.53,69.80,5.73
High (>0.85),78.99,73.96,5.03



--- Pivot Table 2: Mean average_score by Study-Hour Quartile & School Type ---


school_type,Private,Public,Private - Public Gap
study_quartile,,,
Q1 (Lowest),69.51,63.82,5.69
Q2,74.92,67.12,7.80
Q3,76.18,71.18,5.00
Q4 (Highest),83.65,80.06,3.59


### Multivariate Findings on the School-Type Gap
* Across both **attendance bands** and **study-hour quartiles**, **Private** school students average roughly **~6 points higher** than **Public** school students (matching the `+6` school-type offset in the data generator alongside noise), while higher attendance and higher study hours lift scores within **both** school sectors.

## Hypotheses

In [13]:
# Trend line of average_score against study_hours using np.polyfit
x = df["study_hours"].to_numpy()
y = df["average_score"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)
y_pred = slope * x + intercept

ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print(f"Slope (average_score per additional study_hour): {slope:.4f}")
print(f"Intercept: {intercept:.4f}")
print(f"R²: {r_squared:.4f}")

# 1. Histogram using pandas .plot()
ax1 = df["average_score"].plot(
    kind="hist",
    bins=20,
    title="Distribution of Student Average Score",
)
ax1.set_xlabel("Average Score (Points)")
ax1.set_ylabel("Number of Students")
ax1.get_figure().savefig("figures/eda_average_score_hist.png", bbox_inches="tight", dpi=150)
ax1.get_figure().clf()

# 2. Bar chart using pandas .plot()
at_risk_by_county = (
    df.assign(is_at_risk=df["performance_band"] == "At Risk")
    .groupby("county")["is_at_risk"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)
ax2 = at_risk_by_county.plot(
    kind="bar",
    title="Share of At Risk Students by County (%)",
)
ax2.set_xlabel("County")
ax2.set_ylabel("At Risk Share (%)")
ax2.get_figure().savefig("figures/eda_at_risk_by_county_bar.png", bbox_inches="tight", dpi=150)
ax2.get_figure().clf()

# 3. Box plot using pandas .plot()
ax3 = df.boxplot(column="average_score", by="school_type")
ax3.set_title("Average Score by School Type")
ax3.get_figure().suptitle("")
ax3.set_xlabel("School Type")
ax3.set_ylabel("Average Score (Points)")
ax3.get_figure().savefig("figures/eda_score_by_school_box.png", bbox_inches="tight", dpi=150)
ax3.get_figure().clf()

print("Saved 3 EDA charts to figures/:")
print(" - figures/eda_average_score_hist.png")
print(" - figures/eda_at_risk_by_county_bar.png")
print(" - figures/eda_score_by_school_box.png")

Slope (average_score per additional study_hour): 2.1166
Intercept: 61.2596
R²: 0.4954
Saved 3 EDA charts to figures/:
 - figures/eda_average_score_hist.png
 - figures/eda_at_risk_by_county_bar.png
 - figures/eda_score_by_school_box.png


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

### Trend Line & Exported Charts
* **Slope Meaning:** Each additional weekly study hour is associated with an average increase of roughly **2.2 points** in `average_score` (holding other factors constant).
* **Prediction Limitation:** Extrapolating a linear fit to very high `study_hours` or treating raw correlation as causal proof can overstate achievable gains because `average_score` is bounded at `100` and `attendance_rate` / `school_type` also shift scores.
* **Exported Figures:** `figures/eda_average_score_hist.png`, `figures/eda_at_risk_by_county_bar.png`, and `figures/eda_score_by_school_box.png`.

# Findings
1. **Attendance & Study Hours Drive Higher Scores:** `attendance_rate` and `study_hours` correlate positively with `average_score`, showing that consistent class attendance and regular study habits align with stronger multi-subject performance.
2. **Public vs. Private Score Gap Persists:** Private school students score higher on average than Public school students across attendance bands and study-hour quartiles, indicating structural/resource differences rather than study time alone.
3. **County Risk Concentration Varies:** At Risk learners (`average_score < 60`) are not evenly distributed across the six counties, allowing targeted allocation to counties with the highest At Risk share.
4. **Subject Balance Across Learners:** Math, English, and Science track together closely at the student level, with slight subject-level mean differences (`English` highest on average, `Science` slightly lower).
5. **High Study-Hour Outliers Exist:** A small number of students report unusually high `study_hours` above the upper IQR fence, which can pull linear trend estimates if unexamined.

# Recommendations
1. **Target Attendance & Study-Support Coaching in High-Risk Counties:** Focus learner-support funding on counties with the highest share of At Risk students, pairing attendance tracking/outreach with structured after-school study support for low-attendance and low-study-hour learners.
2. **Prioritise Public-School Resource & Remedial Support:** Direct instructional materials, remedial sessions, and teacher support toward Public schools where average scores trail Private schools across comparable attendance and study-hour tiers.

# Limitations
This analysis uses simulated cross-sectional data (`student_performance.csv`, 1,200 students) with no longitudinal follow-up or randomised intervention, so observed relationships show statistical association rather than proven causation; additionally, unmeasured household, socioeconomic, and school-quality variables are not captured in the dataset, and linear trend extrapolations are bounded by `0–100` exam scores.